# Vaidya — Phase 2: QLoRA Training (Lightning.ai)

**GPU:** A10G 24GB — native BF16  
**Expected runtime:** ~2.5–3 hrs (full epoch)

Assumes the full Vaidya repo is already cloned in `/teamspace/studios/this_studio/`.

## Step 1 — Install packages

In [ ]:
!pip install -q \
    transformers==4.47.0 \
    peft==0.14.0 \
    trl==0.15.2 \
    accelerate==0.34.2 \
    datasets==3.2.0 \
    mlflow==2.19.0 \
    boto3 \
    sentencepiece \
    "fsspec[http]==2024.9.0" \
    "packaging==24.2"

# Force-reinstall bitsandbytes so it picks up the CUDA library, not the CPU fallback
!pip install -q bitsandbytes --upgrade --force-reinstall

# Flash Attention 2 — try each known CUDA path until one works
import subprocess, os
cuda_homes = ['/usr/local/cuda', '/usr/local/cuda-12', '/usr/local/cuda-11', '/usr/cuda']
installed = False
for cuda_home in cuda_homes:
    if os.path.exists(cuda_home):
        env = {**os.environ, 'CUDA_HOME': cuda_home, 'PATH': f"{cuda_home}/bin:{os.environ['PATH']}"}
        r = subprocess.run(['pip', 'install', 'flash-attn', '--no-build-isolation', '-q'],
                           env=env, capture_output=True)
        if r.returncode == 0:
            print(f'✅ flash-attn installed via {cuda_home}')
            installed = True
            break
if not installed:
    print('flash-attn not available — using eager attention')

print('✅ Install done')
print()
print('⚠️  RESTART THE KERNEL NOW, then run from Step 2.')

## Step 2 — Verify GPU

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU — attach an A10G in Studio settings.'

gpu  = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
cc   = torch.cuda.get_device_capability()

print(f'GPU  : {gpu}')
print(f'VRAM : {vram:.1f} GB')
print(f'CC   : {cc[0]}.{cc[1]}')
print(f'BF16 native : {cc[0] >= 8}')
print(f'Flash Attn  : ', end='')
try:
    import flash_attn; print(f'✅ {flash_attn.__version__}')
except ImportError:
    print('not installed (training still works)')

## Step 3 — Set working directory & verify scripts

In [ ]:
import os, importlib.util

work_dir = '/teamspace/studios/this_studio/vaidya/layer2_finetune/lightning'
os.chdir(work_dir)
print(f'Working dir: {os.getcwd()}')

for fname, modname in [('train.py', 'train_mod'), ('qlora_config.py', 'qlora_config')]:
    spec = importlib.util.spec_from_file_location(modname, f'{work_dir}/{fname}')
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    print(f'{fname}  →  version {getattr(mod, "__version__", "MISSING")}')

## Step 4 — AWS credentials

In [ ]:
import os, boto3

# ── Fill in your credentials ────────────────────────────────
os.environ['AWS_ACCESS_KEY_ID']     = 'YOUR_ACCESS_KEY_ID'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'YOUR_SECRET_ACCESS_KEY'
os.environ['AWS_DEFAULT_REGION']    = 'ap-south-1'
# ────────────────────────────────────────────────────────────

s3 = boto3.client('s3')
buckets = [b['Name'] for b in s3.list_buckets()['Buckets']]
print('Buckets:', buckets)
assert 'vaidya-artifacts' in buckets, '❌ vaidya-artifacts not found — check credentials'

## Step 5 — Pull data from S3

In [ ]:
import boto3, os, pandas as pd

os.makedirs('./data', exist_ok=True)
s3 = boto3.client('s3')

for page in s3.get_paginator('list_objects_v2').paginate(Bucket='vaidya-artifacts', Prefix='data/'):
    for obj in page.get('Contents', []):
        key = obj['Key']
        if key.endswith('.parquet'):
            fname = key.split('/')[-1]
            local = f'./data/{fname}'
            if not os.path.exists(local):
                print(f'Downloading {key}...')
                s3.download_file('vaidya-artifacts', key, local)
            else:
                print(f'Already exists: {local}')

for split in ['train', 'val', 'test']:
    df = pd.read_parquet(f'./data/{split}.parquet')
    print(f'{split}: {len(df):,} rows')

## Step 6 — Pre-flight check

In [ ]:
import importlib

required = ['transformers', 'peft', 'trl', 'bitsandbytes', 'accelerate', 'datasets', 'mlflow', 'boto3']
all_ok = True
for pkg in required:
    try:
        m = importlib.import_module(pkg)
        print(f'✅  {pkg} {getattr(m, "__version__", "?")}')
    except ImportError as e:
        print(f'❌  {pkg}  — {e}')
        all_ok = False

if not all_ok:
    print('\n⚠️  Re-run Step 1 and restart the kernel.')
else:
    print('\n✅ Environment ready — proceed to training.')

## Step 7 — Run training

In [ ]:
!python train.py --help

In [ ]:
# ⚡ FULL TRAINING RUN — A100 80GB ~40–50 min
# run2 avoids S3 collision with Kaggle run1
!python train.py --run-name run2

## Step 8 — Verify checkpoint on S3 (or manually upload if training was interrupted)

In [ ]:
# import boto3, os, glob
# from pathlib import Path

# run_name = 'run3'   # change if different
# ckpt_root = Path(f'./checkpoints/{run_name}')

# # Show what's saved locally
# print('=== Local checkpoints ===')
# ckpts = sorted(glob.glob(str(ckpt_root / 'checkpoint-*')))
# if ckpts:
#     for c in ckpts:
#         files = list(Path(c).rglob('*'))
#         size_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
#         print(f'  {c}  ({size_mb:.0f} MB)')
# elif ckpt_root.exists():
#     print(f'  {ckpt_root} exists but no checkpoint-* subdirs')
#     files = list(ckpt_root.rglob('*'))
#     size_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
#     print(f'  Final checkpoint: {size_mb:.0f} MB total')
# else:
#     print(f'  ❌ {ckpt_root} not found')

# print()

# # Upload everything to S3
# print('=== Uploading to S3 ===')
# s3 = boto3.client('s3')
# bucket = 'vaidya-artifacts'
# uploaded = 0
# for f in ckpt_root.rglob('*'):
#     if f.is_file():
#         key = f'checkpoints/{run_name}/{f.relative_to(ckpt_root).as_posix()}'
#         s3.upload_file(str(f), bucket, key)
#         uploaded += 1
#         if uploaded % 5 == 0:
#             print(f'  uploaded {uploaded} files...')

# print(f'✅ Done — {uploaded} files → s3://vaidya-artifacts/checkpoints/{run_name}/')

=== Local checkpoints ===
  ❌ checkpoints\run3 not found

=== Uploading to S3 ===
✅ Done — 0 files → s3://vaidya-artifacts/checkpoints/run3/


## Step 9 — Quick loss check

In [6]:
import json, glob

states = glob.glob('./checkpoints/run3/checkpoint-*/trainer_state.json')
if not states:
    states = glob.glob('./checkpoints/run3/trainer_state.json')

if states:
    with open(sorted(states)[-1]) as f:
        state = json.load(f)
    log_history = state.get('log_history', [])
    train_losses = [(e['step'], e['loss']) for e in log_history if 'loss' in e]
    if train_losses:
        print(f'Train loss — first: {train_losses[0][1]:.4f}  last: {train_losses[-1][1]:.4f}')
        print('✅ Loss decreased.' if train_losses[-1][1] < train_losses[0][1] else '⚠️ Loss did not decrease.')
else:
    print('No trainer_state.json — check ./checkpoints/run3/')

No trainer_state.json — check ./checkpoints/run3/


## Gate check — before Phase 3

- ✅ Training loss decreased
- ✅ Checkpoint at `s3://vaidya-artifacts/checkpoints/run1/`
- ✅ MLflow runs at `s3://vaidya-artifacts/mlflow/`

**Next:** Phase 3 — accuracy eval (base Mistral-7B vs fine-tuned)